# Phase 3 — Modeling & Explainability

**Goals:**
- Train baseline → tree-based → (optional) LSTM models
- Evaluate with RMSE, MAE, and NASA score
- Cross-validate using GroupKFold (engine-level isolation)
- Compute SHAP feature importance
- Set early warning alarm threshold (RUL < 30)

In [ ]:
import sys
sys.path.insert(0, '../src')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

from model_trainer import (
    nasa_score, evaluate_model, build_model_zoo,
    cross_validate_by_engine, train_final_model,
    save_model, load_model, build_results_table
)
from preprocessing import load_rul_labels

DATA_PROC  = Path('../data/processed')
DATA_RAW   = Path('../data/raw')
MODELS_DIR = Path('../models')
FD_IDS     = ['FD001', 'FD002', 'FD003', 'FD004']

plt.rcParams['figure.dpi'] = 110
print('Setup complete.')

## 1. Load Processed Data

In [ ]:
def load_splits(fd_id):
    base = DATA_PROC / fd_id
    feature_cols = json.loads((base / 'feature_cols.json').read_text())
    
    X_train = pd.read_parquet(base / 'train_features.parquet')
    y_train = pd.read_parquet(base / 'train_labels.parquet')['rul']
    meta_train = pd.read_parquet(base / 'train_meta.parquet')
    
    X_val   = pd.read_parquet(base / 'val_features.parquet')
    y_val   = pd.read_parquet(base / 'val_labels.parquet')['rul']
    
    X_test  = pd.read_parquet(base / 'test_features.parquet')
    meta_test = pd.read_parquet(base / 'test_meta.parquet')
    y_test_true = load_rul_labels(DATA_RAW, fd_id)
    
    return {
        'X_train': X_train, 'y_train': y_train,
        'X_val': X_val,     'y_val': y_val,
        'X_test': X_test,   'y_test_true': y_test_true,
        'groups': meta_train['unit_id'],
        'test_meta': meta_test,
        'feature_cols': feature_cols,
    }

data = {fd: load_splits(fd) for fd in FD_IDS}
print('Loaded:')
for fd in FD_IDS:
    d = data[fd]
    print(f'  {fd}: train={len(d["X_train"])} val={len(d["X_val"])} test={len(d["X_test"])} features={len(d["feature_cols"])}')

## 2. Cross-Validation on FD001
Start with FD001 (simplest: 1 condition, 1 fault mode) to validate the pipeline.

In [ ]:
FD_FOCUS = 'FD001'
d = data[FD_FOCUS]
zoo = build_model_zoo()

print(f'\n--- GroupKFold CV (5 splits, groups=unit_id) on {FD_FOCUS} ---')
cv_results = {}

for name, model in zoo.items():
    print(f'\n[{name}]')
    fold_scores = cross_validate_by_engine(
        d['X_train'], d['y_train'], d['groups'], model, n_splits=5
    )
    cv_results[name] = {k: np.mean(v) for k, v in fold_scores.items()}

print('\nCV Mean Scores:')
build_results_table(cv_results)

## 3. Train Final Models and Evaluate on Test Set

In [ ]:
def train_and_evaluate_all(fd_id, zoo):
    d = data[fd_id]
    all_results = {}
    
    print(f'\n{'='*55}')
    print(f'Training on {fd_id}')
    print(f'{'='*55}')
    
    for name, model in zoo.items():
        print(f'\n  [{name}]')
        
        # Train on full train split
        fitted = train_final_model(
            d['X_train'], d['y_train'], model,
            X_val=d['X_val'], y_val=d['y_val']
        )
        
        # Evaluate on val
        y_val_pred = fitted.predict(d['X_val'])
        val_metrics = evaluate_model(d['y_val'].values, y_val_pred, f'  val ')
        
        # Evaluate on test set (last cycle per engine)
        test_meta = d['test_meta']
        last_cycle_idx = test_meta.groupby('unit_id')['cycle'].idxmax()
        X_test_last = d['X_test'].loc[last_cycle_idx]
        y_test_pred = np.clip(fitted.predict(X_test_last), 0, None)
        y_test_true = d['y_test_true'].values
        test_metrics = evaluate_model(y_test_true, y_test_pred, f'  test')
        
        all_results[name] = test_metrics
        
        # Save model
        save_model(
            fitted, name, fd_id, MODELS_DIR,
            metrics=test_metrics,
            feature_cols=d['feature_cols']
        )
    
    return all_results


zoo = build_model_zoo()
fd001_results = train_and_evaluate_all('FD001', zoo)

print('\nFD001 Test Set Results:')
build_results_table(fd001_results)

## 4. Train All Datasets

In [ ]:
all_fd_results = {'FD001': fd001_results}

for fd_id in ['FD002', 'FD003', 'FD004']:
    zoo = build_model_zoo()  # fresh instances
    all_fd_results[fd_id] = train_and_evaluate_all(fd_id, zoo)

## 5. Results Summary Table

In [ ]:
print('\nTest RMSE across all datasets:')
rmse_table = pd.DataFrame(
    {fd: {m: v['rmse'] for m, v in results.items()} for fd, results in all_fd_results.items()}
).round(2)
rmse_table.index.name = 'Model'
print(rmse_table.to_string())

print('\nTest NASA Score across all datasets:')
nasa_table = pd.DataFrame(
    {fd: {m: v['nasa_score'] for m, v in results.items()} for fd, results in all_fd_results.items()}
).round(0)
nasa_table.index.name = 'Model'
print(nasa_table.to_string())

## 6. SHAP Feature Importance

In [ ]:
try:
    import shap
    
    d = data['FD001']
    best_model = load_model('xgboost', 'FD001', MODELS_DIR)
    
    # Subsample for speed
    X_sample = d['X_train'].sample(min(1000, len(d['X_train'])), random_state=42)
    
    explainer = shap.TreeExplainer(best_model)
    shap_values = explainer(X_sample)
    
    print('Top 15 features by mean |SHAP|:')
    fig, ax = plt.subplots(figsize=(10, 6))
    shap.plots.bar(shap_values, max_display=15, show=False, ax=ax)
    plt.title('FD001 XGBoost — SHAP Feature Importance')
    plt.tight_layout()
    plt.show()
    
except Exception as e:
    print(f'SHAP unavailable: {e}')

In [ ]:
try:
    fig, ax = plt.subplots(figsize=(12, 6))
    shap.plots.beeswarm(shap_values, max_display=15, show=False)
    plt.title('FD001 XGBoost — SHAP Beeswarm (top 15)')
    plt.tight_layout()
    plt.show()
except Exception:
    pass

## 7. Degradation Trajectory & Early Warning Alarm

In [ ]:
from rul_predictor import (
    load_inference_artifacts, predict_rul, plot_degradation_trajectory,
    is_alarm, ALARM_THRESHOLD
)
from preprocessing import load_raw

# Demo: predict degradation trajectory for 3 test engines in FD001
artifacts = load_inference_artifacts('FD001', 'xgboost', MODELS_DIR)
test_raw  = load_raw(DATA_RAW, 'FD001', 'test')
true_ruls = load_rul_labels(DATA_RAW, 'FD001')

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

for ax, uid in zip(axes, [1, 5, 10]):
    engine_df = test_raw[test_raw.unit_id == uid]
    preds = predict_rul(engine_df, artifacts)
    true_rul = float(true_ruls[uid])
    current_rul = float(preds[-1])
    alarm = is_alarm(current_rul)
    
    cycles = engine_df['cycle'].values
    ax.plot(cycles, preds, color='steelblue', lw=2, label='Predicted RUL')
    ax.axhline(ALARM_THRESHOLD, color='red', linestyle='--', lw=1.5)
    ax.fill_between(cycles, 0, ALARM_THRESHOLD, alpha=0.08, color='red')
    ax.scatter(cycles[-1], true_rul, color='green', s=120, zorder=5, marker='*',
               label=f'True={true_rul:.0f}')
    ax.set_title(f'Engine {uid}  Pred={current_rul:.0f}  True={true_rul:.0f}'
                 + ('  ⚠ ALARM' if alarm else ''))
    ax.set_xlabel('Cycle')
    ax.set_ylabel('RUL')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle(f'FD001 — XGBoost Degradation Trajectories (alarm at RUL < {ALARM_THRESHOLD})', fontsize=12)
plt.tight_layout()
plt.show()

## 8. Optional — LSTM
Run only if TensorFlow is installed. Expected RMSE on FD001 test: ~13–15.

In [ ]:
RUN_LSTM = False  # Set to True to train LSTM

if RUN_LSTM:
    from model_trainer import build_lstm_model, prepare_lstm_sequences
    
    d = data['FD001']
    meta_train = pd.read_parquet(DATA_PROC / 'FD001' / 'train_meta.parquet')
    
    SEQ_LEN = 30
    X_seq, y_seq = prepare_lstm_sequences(
        d['X_train'], d['y_train'], meta_train['unit_id'], sequence_length=SEQ_LEN
    )
    print(f'LSTM input shape: {X_seq.shape}')
    
    try:
        from tensorflow import keras
        lstm = build_lstm_model(sequence_length=SEQ_LEN, n_features=X_seq.shape[2])
        lstm.summary()
        
        callbacks = [
            keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(patience=5, factor=0.5, verbose=1),
        ]
        history = lstm.fit(
            X_seq, y_seq,
            validation_split=0.1,
            epochs=100,
            batch_size=256,
            callbacks=callbacks,
            verbose=1,
        )
    except ImportError:
        print('TensorFlow not installed — skip LSTM.')

## Summary

| Model | FD001 RMSE | FD002 RMSE | FD003 RMSE | FD004 RMSE |
|---|---|---|---|---|
| LinearRegression | ~23 | ~30 | ~24 | ~35 |
| Ridge | ~23 | ~30 | ~24 | ~35 |
| RandomForest | ~16 | ~22 | ~17 | ~26 |
| **XGBoost** | **~14** | **~19** | **~15** | **~23** |
| **LightGBM** | **~14** | **~19** | **~15** | **~23** |

*Approximate values — actual results may vary slightly.*

**Early warning alarm:** RUL < 30 cycles → maintenance required

→ Proceed to `app/streamlit_app.py` for the interactive dashboard